### Imports & Data Loading


In [2]:
import numpy as np
from sklearn.datasets import load_breast_cancer

np.random.seed(42)

data = load_breast_cancer()

X = data.data
y = data.target

### Stratified Train/Test Split


In [3]:
def train_test_split(X, y, test_size=0.2, seed=42):
    rng = np.random.default_rng(seed)

    train_indices = []
    test_indices = []

    for label in np.unique(y):
        indices = np.where(y == label)[0]
        rng.shuffle(indices)

        n_test = int(round(len(indices) * test_size))

        test_indices.extend(indices[:n_test])
        train_indices.extend(indices[n_test:])

    train_indices = np.array(train_indices)
    test_indices = np.array(test_indices)

    rng.shuffle(train_indices)
    rng.shuffle(test_indices)

    return (
        X[train_indices],
        X[test_indices],
        y[train_indices],
        y[test_indices]
    )

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("Training class counts:", np.bincount(y_train))
print("Testing class counts:", np.bincount(y_test))

Training shape: (456, 30)
Testing shape: (113, 30)
Training class counts: [170 286]
Testing class counts: [42 71]


### Feature Standardization



In [5]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

std[std == 0] = 1

X_train_scaled = (X_train - mean) / std
X_test_scaled = (X_test - mean) / std

### Gaussian Naive Bayes 
$$P(C | x) \propto P(C) \cdot \prod_i P(x_i | C)$$



In [6]:
class GaussianNaiveBayes:

    def __init__(self, var_smoothing=1e-9):
        self.var_smoothing = var_smoothing

    def fit(self, X, y):
        self.classes = np.unique(y)

        self.theta_ = np.array([
            X[y == c].mean(axis=0)
            for c in self.classes
        ])

        self.var_ = np.array([
            X[y == c].var(axis=0)
            for c in self.classes
        ])

        counts = np.array([
            np.sum(y == c)
            for c in self.classes
        ])

        self.class_prior_ = counts / len(y)

        epsilon = (
            self.var_smoothing
            * np.var(X, axis=0).max()
        )

        self.epsilon_ = epsilon
        self.var_smoothed_ = self.var_ + epsilon

        return self

    def predict_proba(self, X):
        log_probs = []

        for i, c in enumerate(self.classes):
            mean = self.theta_[i]
            var = self.var_smoothed_[i]

            log_likelihood = -0.5 * np.sum(
                np.log(2 * np.pi * var)
                + ((X - mean) ** 2) / var,
                axis=1
            )

            log_prior = np.log(self.class_prior_[i])

            log_probs.append(log_likelihood + log_prior)

        log_probs = np.array(log_probs).T

        # Numerical stability
        log_probs -= log_probs.max(axis=1, keepdims=True)

        probabilities = np.exp(log_probs)

        return probabilities / probabilities.sum(
            axis=1, keepdims=True
        )

    def predict(self, X):
        probabilities = self.predict_proba(X)

        indices = np.argmax(probabilities, axis=1)

        return self.classes[indices]


In [7]:
model = GaussianNaiveBayes()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = np.mean(y_pred == y_test)

print("\n--- Without Standardization ---")
print("Accuracy:", accuracy)
print("Accuracy (%):", accuracy * 100)


--- Without Standardization ---
Accuracy: 0.9380530973451328
Accuracy (%): 93.80530973451327


In [8]:
scaled_model = GaussianNaiveBayes()
scaled_model.fit(X_train_scaled, y_train)

scaled_pred = scaled_model.predict(X_test_scaled)

scaled_accuracy = np.mean(scaled_pred == y_test)

print("\n--- With Standardization ---")
print("Accuracy:", scaled_accuracy)
print("Accuracy (%):", scaled_accuracy * 100)


--- With Standardization ---
Accuracy: 0.9292035398230089
Accuracy (%): 92.92035398230088


### Sanity Check 

In [9]:
X_tiny = np.array([
    [1.0],
    [3.0],
    [5.0],
    [7.0]
])

y_tiny = np.array([0, 0, 1, 1])

tiny_model = GaussianNaiveBayes(var_smoothing=0)
tiny_model.fit(X_tiny, y_tiny)

print("\n--- Sanity Test ---")
print("Classes:", tiny_model.classes)
print("Means:", tiny_model.theta_.flatten())
print("Variances:", tiny_model.var_.flatten())
print("Class priors:", tiny_model.class_prior_)

sample = np.array([[2.0]])

print("Sample:", sample.flatten())
print("Posterior probabilities:",
      tiny_model.predict_proba(sample))
print("Predicted class:", tiny_model.predict(sample)[0])



--- Sanity Test ---
Classes: [0 1]
Means: [2. 6.]
Variances: [1. 1.]
Class priors: [0.5 0.5]
Sample: [2.]
Posterior probabilities: [[9.9966465e-01 3.3535013e-04]]
Predicted class: 0
